# 01 — Data availability audit (M1.1)

Inventory of every Sentinel-2 L2A (PC `sentinel-2-l2a`) and Landsat Collection 2 L2 (PC `landsat-c2-l2`)
acquisition over `data/aoi/aoi.geojson`, with the share of AOI pixels that are clear on each, from the native-grid
**SCL** (20 m) and **QA_PIXEL** (30 m) bands. No reflectance is read, so the Sentinel-2 offset and Landsat scaling
do not apply here. All logic is in `highland.stac`, `highland.masking` and `highland.inventory`; this notebook only
orchestrates, tabulates and plots. Definitions are in ADR 0010; the window proposal is ADR 0011.

Clear-pixel metric: the **sum over acquisitions of the clear fraction** in a month, which equals the expected number
of clear observations per AOI pixel. It needs no threshold. Results are written to `outputs/M1.1/`.

In [ ]:
import json
import logging
import time
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from highland import inventory, stac
from highland.aoi import aoi_area_km2, load_aoi

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "outputs" / "M1.1"
OUT.mkdir(parents=True, exist_ok=True)
RUN_DATE = date(
    2026, 11, 2
)  # frozen search end: reruns reproduce the same item search (cached JSON)
RANGES = {  # search ranges (ISO dates, UTC); S2 archive starts 2015-06-27, Landsat 1982-08-22
    "s2": ("2015-06-01", RUN_DATE.isoformat()),
    "landsat": ("1982-01-01", RUN_DATE.isoformat()),
}
COLLECTIONS = {"s2": stac.S2_COLLECTION, "landsat": stac.LANDSAT_COLLECTION}
PILOT = ("2023-01-01", "2023-12-31")
timings: dict[str, float] = {}

aoi = load_aoi()
print(f"AOI {aoi_area_km2(aoi):.1f} km^2 from data/aoi/aoi.geojson; run date {RUN_DATE}")

## 1. Pilot: one year (2023) of each sensor
Times the audit and warms the resume cache (the full run reuses it).

In [ ]:
pilot = {}
for sensor in ("s2", "landsat"):
    t0 = time.perf_counter()
    items = stac.search_items(COLLECTIONS[sensor], aoi, *PILOT)
    n_raw = len(items)
    if sensor == "s2":
        items, _ = stac.dedupe_s2(items)
    t_search = time.perf_counter() - t0
    t0 = time.perf_counter()
    df_p = inventory.run_audit(sensor, items, aoi, OUT)
    t_audit = time.perf_counter() - t0
    pilot[sensor] = {
        "items_raw": n_raw,
        "items_dedup": len(items),
        "acquisitions": len(df_p),
        "errors": df_p.attrs["n_error"],
        "search_s": round(t_search, 1),
        "audit_s": round(t_audit, 1),
    }
    timings[f"pilot_{sensor}_s"] = round(t_search + t_audit, 1)
pd.DataFrame(pilot).T

## 2. Full search, deduplication and item counts

In [ ]:
cache = OUT / "cache"
raw: dict[str, list] = {}
items_all: dict[str, list] = {}
dropped = pd.DataFrame()
for sensor in ("s2", "landsat"):
    path = cache / f"items_{sensor}_{RANGES[sensor][0]}_{RANGES[sensor][1]}.json"
    t0 = time.perf_counter()
    if path.exists():
        raw[sensor] = stac.load_items(path)
    else:
        raw[sensor] = stac.search_items(COLLECTIONS[sensor], aoi, *RANGES[sensor])
        stac.save_items(raw[sensor], path)
    timings[f"search_{sensor}_s"] = round(time.perf_counter() - t0, 1)
    if sensor == "s2":
        items_all[sensor], dropped = stac.dedupe_s2(raw[sensor])
    else:
        items_all[sensor] = raw[sensor]
    print(f"{sensor}: {len(raw[sensor])} items found, {len(items_all[sensor])} after deduplication")
dropped.to_csv(OUT / "dedupe_dropped.csv", index=False)
print("S2 items dropped:", len(dropped), dropped["reason"].value_counts().to_dict())

In [ ]:
s2_before = stac.item_table(raw["s2"], aoi)
s2_after = stac.item_table(items_all["s2"], aoi)
ls_items = stac.item_table(items_all["landsat"], aoi)
tbl_before = (
    s2_before.groupby(["platform", "baseline"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
)
tbl_after = (
    s2_after.groupby(["platform", "baseline"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
)
print("Sentinel-2 items by platform x baseline")
print("before deduplication")
display(tbl_before)
print("after deduplication")
display(tbl_after)
print("Dropped duplicates (reason x kept baseline x dropped baseline)")
display(dropped.groupby(["reason", "kept_baseline", "dropped_baseline"]).size())
print("Sentinel-2 MGRS tiles (after dedupe): items and mean AOI cover of the item footprint")
display(
    s2_after.groupby("tile").agg(items=("id", "size"), aoi_cover_mean=("aoi_cover_frac", "mean"))
)
print("Landsat items by platform x tier")
display(
    ls_items.groupby(["platform", "collection_category"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
)
print("Landsat WRS path/rows: items and mean AOI cover")
display(
    ls_items.groupby("tile").agg(items=("id", "size"), aoi_cover_mean=("aoi_cover_frac", "mean"))
)
tbl_before.to_csv(OUT / "s2_items_before_dedupe.csv")
tbl_after.to_csv(OUT / "s2_items_after_dedupe.csv")

## 3. Audit every acquisition
One acquisition at a time; resumable (`outputs/M1.1/cache/`), failed reads kept as `status=error` rows.

In [ ]:
frames = {}
for sensor in ("s2", "landsat"):
    t0 = time.perf_counter()
    frames[sensor] = inventory.run_audit(sensor, items_all[sensor], aoi, OUT)
    timings[f"audit_{sensor}_s"] = round(time.perf_counter() - t0, 1)
    print(
        sensor,
        len(frames[sensor]),
        "acquisitions,",
        frames[sensor].attrs["n_error"],
        "errors,",
        f"{timings[f'audit_{sensor}_s']} s this run",
    )
inv = pd.concat(frames.values(), ignore_index=True)
inventory.write_inventory(inv, OUT / "scene_inventory.parquet")
errs = inv[inv["status"] == "error"][["sensor", "acq_id", "error"]]
print("Error rows:", len(errs))
display(errs)
display(inv.groupby(["sensor", "platform", "status"]).size().unstack(fill_value=0))

In [ ]:
s2 = inv[(inv["sensor"] == "s2") & (inv["status"] == "ok")].copy()
ls = inv[(inv["sensor"] == "landsat") & (inv["status"] == "ok")].copy()
s2["era"] = inventory.baseline_era(s2)
print("Sentinel-2 acquisitions (platform + local day; tiles of one pass fused)")
display(
    s2.groupby(["platform", "baseline_max"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
)
print(
    "Acquisitions with mixed baselines (fused tiles at different baselines):",
    int(s2["mixed_baseline"].sum()),
)
print("Landsat acquisitions by platform x tier (relative_orbit-style '+'-joined strings)")
display(
    ls.groupby(["platform", "collection_category"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
)
print("Landsat 7 SLC-off acquisitions:", int(ls["slc_off"].sum()))
l7 = ls[ls["platform"] == "landsat-7"]
print(
    "Landsat 7 valid_frac: all",
    round(l7["valid_frac"].mean(), 3),
    "min",
    round(l7["valid_frac"].min(), 3),
    "| SLC-off only mean",
    round(l7.loc[l7["slc_off"], "valid_frac"].mean(), 3),
)
t2 = ls[ls["collection_category"].str.contains("T2")]
print(
    "Landsat T2 acquisitions:",
    len(t2),
    "; with clear == 0:",
    int((t2["clear"] == 0).sum()),
    "; T1 mean clear",
    round(ls.loc[~ls.index.isin(t2.index), "clear"].mean(), 3),
)
print(
    "Acquisitions with valid_frac < 1 (no data over part of the AOI): S2",
    int((s2["valid_frac"] < 1).sum()),
    "Landsat",
    int((ls["valid_frac"] < 1).sum()),
    "; with no data at all: S2",
    int((s2["valid_frac"] == 0).sum()),
    "Landsat",
    int((ls["valid_frac"] == 0).sum()),
)
print("S2 acquisitions by relative orbit (item footprints can reach the AOI without data)")
display(
    s2.groupby("relative_orbit").agg(
        acquisitions=("acq_id", "size"), no_data=("valid_frac", lambda v: int((v == 0).sum()))
    )
)
display(
    s2.loc[
        s2["valid_frac"] < 1, ["date_local", "platform", "relative_orbit", "tiles", "valid_frac"]
    ]
)
print(
    "First S2 acquisition with data over the AOI:",
    s2.loc[s2["valid_frac"] > 0, "date_local"].min(),
    "| first Landsat:",
    ls["date_local"].min(),
)
print("Acquisitions with data per year, by platform")
display(
    inv[(inv["status"] == "ok") & (inv["valid_frac"] > 0)]
    .groupby(["sensor", "year", "platform"])
    .size()
    .unstack("platform", fill_value=0)
)

## 4. SCL class 2 (cast shadow) and class 7 (unclassified), reported separately
Class 2 meant "dark features" before baseline 04.00 (dark soils, burn scars) and is cast/topographic shadow after.
Rows are split with `baseline_min`/`baseline_max`; acquisitions that fused tiles of different eras are `mixed`.

In [ ]:
share_cols = [
    "scl_share_0",
    "scl_share_2",
    "scl_share_7",
    "scl_share_3",
    "scl_share_8",
    "scl_share_9",
    "scl_share_10",
]
by_era = s2.groupby("era").agg(
    n=("acq_id", "size"),
    class2_mean=("scl_share_2", "mean"),
    class2_median=("scl_share_2", "median"),
    class2_p90=("scl_share_2", lambda v: v.quantile(0.9)),
    class7_mean=("scl_share_7", "mean"),
    class7_median=("scl_share_7", "median"),
    class7_p90=("scl_share_7", lambda v: v.quantile(0.9)),
)
overall = pd.DataFrame(
    {
        "n": [len(s2)],
        "class2_mean": [s2["scl_share_2"].mean()],
        "class2_median": [s2["scl_share_2"].median()],
        "class2_p90": [s2["scl_share_2"].quantile(0.9)],
        "class7_mean": [s2["scl_share_7"].mean()],
        "class7_median": [s2["scl_share_7"].median()],
        "class7_p90": [s2["scl_share_7"].quantile(0.9)],
    },
    index=["all"],
)
s2d = s2[s2["valid_frac"] > 0]  # excluding the acquisitions with no data over the AOI
by_era_d = s2d.groupby("era").agg(
    n=("acq_id", "size"),
    class2_mean=("scl_share_2", "mean"),
    class2_median=("scl_share_2", "median"),
    class2_p90=("scl_share_2", lambda v: v.quantile(0.9)),
    class7_mean=("scl_share_7", "mean"),
    class7_median=("scl_share_7", "median"),
    class7_p90=("scl_share_7", lambda v: v.quantile(0.9)),
)
overall_d = pd.DataFrame(
    {
        "n": [len(s2d)],
        "class2_mean": [s2d["scl_share_2"].mean()],
        "class2_median": [s2d["scl_share_2"].median()],
        "class2_p90": [s2d["scl_share_2"].quantile(0.9)],
        "class7_mean": [s2d["scl_share_7"].mean()],
        "class7_median": [s2d["scl_share_7"].median()],
        "class7_p90": [s2d["scl_share_7"].quantile(0.9)],
    },
    index=["all"],
)
print("Excluding acquisitions with no data over the AOI (valid_frac == 0):")
display(pd.concat([by_era_d, overall_d]))
pd.concat([by_era_d, overall_d]).to_csv(OUT / "scl_class2_class7_shares_with_data.csv")
print("Mean AOI share per SCL class, with-data acquisitions only")
display(s2d[[f"scl_share_{c}" for c in range(12)]].mean().rename("mean_share").to_frame().T)
print(
    "Mean clear fraction by keep-set, with-data acquisitions only:",
    s2d[list(inventory.S2_VARIANTS)].mean().round(4).to_dict(),
)
print("Class 2 = SCL cast shadow only, not self-shading or low illumination.")
class_tbl = pd.concat([by_era, overall])
display(class_tbl)
class_tbl.to_csv(OUT / "scl_class2_class7_shares.csv")
print("Mean AOI share per SCL class (all S2 acquisitions)")
display(s2[[f"scl_share_{c}" for c in range(12)]].mean().rename("mean_share").to_frame().T)
print("Mean clear fraction by keep-set:", s2[list(inventory.S2_VARIANTS)].mean().round(4).to_dict())

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
colors = {"pre-04.00": "tab:orange", "post-04.00": "tab:blue", "mixed": "tab:green"}
for ax, c in zip(axes, (2, 7), strict=True):
    for era, g in s2.groupby("era"):
        ax.scatter(g["datetime_utc"], g[f"scl_share_{c}"] * 100, s=6, c=colors[era], label=era)
    ax.set_ylabel(f"SCL class {c}, % of AOI")
axes[0].legend(title="baseline")
axes[0].set_title("Share of AOI pixels in SCL class 2 (top) and class 7 (bottom), per acquisition")
fig.tight_layout()
fig.savefig(OUT / "scl_class2_class7_shares.png", dpi=120)
plt.show()

## 5. Clear-pixel plots, month x year
Summed clear fraction = expected clear observations per AOI pixel; plus acquisition counts.

In [ ]:
def heatmap(ax, tbl, title, fmt="{:.1f}"):
    im = ax.imshow(tbl.values, aspect="auto", cmap="viridis", origin="upper")
    ax.set_xticks(range(12), list("JFMAMJJASOND"))
    ax.set_yticks(range(len(tbl.index)), tbl.index)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)


def plot_sensor(df, sensor, variants, fname):
    first = int(df["year"].min())
    yrs = range(first, RUN_DATE.year + 1)
    n = len(variants) + 1
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 0.3 * len(yrs) + 2))
    for ax, v in zip(axes, variants, strict=False):
        heatmap(
            ax,
            inventory.month_year_table(df, v).reindex(yrs, fill_value=0.0),
            f"{sensor}: sum of {v}",
        )
    heatmap(
        axes[-1],
        inventory.month_year_table(df[df["valid_frac"] > 0], "acq_id", "count").reindex(
            yrs, fill_value=0.0
        ),
        f"{sensor}: acquisitions with data over AOI",
    )
    fig.tight_layout()
    fig.savefig(OUT / fname, dpi=110)
    plt.show()


plot_sensor(s2, "S2", ["clear_base", "clear_keep27"], "clear_month_year_s2.png")
plot_sensor(ls, "Landsat", ["clear"], "clear_month_year_landsat.png")
for name, d, v in (("s2", s2, "clear_base"), ("landsat", ls, "clear")):
    inventory.month_year_table(d, v).to_csv(OUT / f"clear_sum_month_year_{name}.csv")
    inventory.month_year_table(d[d["valid_frac"] > 0], "acq_id", "count").to_csv(
        OUT / f"acq_count_month_year_{name}.csv"
    )
print("Expected clear observations per AOI pixel, S2 clear_base (rows year, columns month)")
display(inventory.month_year_table(s2, "clear_base").round(2))

## 5b. Per-acquisition clear rates by window and month
Window totals depend on window length (SWM has 5 months, the others 2 to 3). The mean clear fraction of one acquisition,
over acquisitions with data over the AOI, does not.

In [ ]:
SEASONS = {
    "FIM_Mar-Apr": (3, 4),
    "SWM_May-Sep": (5, 6, 7, 8, 9),
    "SIM_Oct-Nov": (10, 11),
    "NEM_Dec-Feb": (12, 1, 2),
}
rates = {}
for name, d, v in (("s2", s2, "clear_base"), ("landsat", ls, "clear")):
    rw = inventory.clear_rate_by_window(d, v, SEASONS)
    rm = inventory.clear_rate_by_month(d, v)
    rw.to_csv(OUT / f"clear_rate_by_window_{name}.csv", index=False)
    rm.to_csv(OUT / f"clear_rate_by_month_{name}.csv")
    print(name, v, "per-acquisition clear fraction by window")
    display(rw.round(3))
    print(name, v, "per-acquisition clear fraction by calendar month")
    display(rm.round(3))

## 6. Per-pixel clear counts and window tables
Windows are the four Met Department seasons; December belongs to the following year's NE monsoon. `search_range` makes `partial_window` exact.

In [ ]:
geoboxes, yms, counts, inside = {}, {}, {}, {}
for sensor in ("s2", "landsat"):
    t0 = time.perf_counter()
    geoboxes[sensor] = stac.aoi_geobox(aoi, items_all[sensor], sensor)
    inside[sensor] = inventory.aoi_inside_mask(aoi, geoboxes[sensor])
    d = inv[inv["sensor"] == sensor]
    yms[sensor], counts[sensor] = inventory.clear_counts(OUT, sensor, d, geoboxes[sensor])
    for v, arr in counts[sensor].items():
        inventory.write_clear_counts(
            OUT / f"clear_counts_{sensor}_{v}.tif",
            yms[sensor],
            arr,
            geoboxes[sensor],
            inside[sensor],
        )
    timings[f"counts_{sensor}_s"] = round(time.perf_counter() - t0, 1)
    print(
        sensor,
        "grid",
        geoboxes[sensor].shape,
        "px;",
        int(inside[sensor].sum()),
        "AOI pixels;",
        len(yms[sensor]),
        "year-months;",
        list(counts[sensor]),
    )

In [ ]:
def windows_table(sensor, variants, windows, years):
    d = inv[inv["sensor"] == sensor]
    rng = tuple(date.fromisoformat(x) for x in RANGES[sensor])
    parts = []
    for v in variants:
        t = inventory.window_summary(
            d,
            yms[sensor],
            counts[sensor][v],
            inside[sensor],
            windows,
            years,
            today=RUN_DATE,
            search_range=rng,
        )
        parts.append(t.assign(variant=v))
    return pd.concat(parts, ignore_index=True)


s2_years = range(2015, RUN_DATE.year + 1)
ls_years = range(1982, RUN_DATE.year + 1)
win_s2 = windows_table("s2", ["clear_base", "clear_keep2", "clear_keep27"], SEASONS, s2_years)
win_ls = windows_table("landsat", ["clear", "clear_nocirrus"], SEASONS, ls_years)
win_s2.to_csv(OUT / "window_summary_s2.csv", index=False)
win_ls.to_csv(OUT / "window_summary_landsat.csv", index=False)


def pivot(win, variant, col):
    t = win[win["variant"] == variant].pivot(index="season_year", columns="window", values=col)
    return t[list(SEASONS)]


print("S2 (clear_base): share of AOI pixels with ZERO clear observations, by season-year")
display(pivot(win_s2, "clear_base", "share_zero_obs").round(3))
print("S2 (clear_base): median clear observations per pixel")
display(pivot(win_s2, "clear_base", "median_count"))
print(
    "S2 (clear_base): lower tail: share of pixels with <= 1 observation, <= 2, and p10 of the count"
)
for col in ("share_le1_obs", "share_le2_obs", "p10_count"):
    print(col)
    display(pivot(win_s2, "clear_base", col).round(3))
print("S2 (clear_base): acquisitions with data (valid_frac > 0)")
display(pivot(win_s2, "clear_base", "n_with_data"))
print("S2: complete (non-partial) season-years only; summary across years")


def across_years(win, variant):
    w = win[(win["variant"] == variant) & ~win["partial_window"]]
    return (
        w.groupby("window")
        .agg(
            years=("season_year", "size"),
            zero_share_median=("share_zero_obs", "median"),
            zero_share_min=("share_zero_obs", "min"),
            zero_share_max=("share_zero_obs", "max"),
            median_count_median=("median_count", "median"),
            median_count_min=("median_count", "min"),
            median_count_max=("median_count", "max"),
            years_median_ge1=("median_count", lambda v: int((v >= 1).sum())),
        )
        .loc[list(SEASONS)]
    )


for v in ("clear_base", "clear_keep2", "clear_keep27"):
    print(v)
    display(across_years(win_s2, v).round(3))
    across_years(win_s2, v).to_csv(OUT / f"window_across_years_s2_{v}.csv")
print(
    "Partial windows (S2):",
    sorted(
        set(
            map(tuple, win_s2.loc[win_s2.partial_window, ["window", "season_year"]].values.tolist())
        )
    ),
)

In [ ]:
print(
    "Landsat (clear): share with ZERO clear observations, by season-year (all years with any row)"
)
display(pivot(win_ls, "clear", "share_zero_obs").round(3))
print("Landsat (clear): median clear observations per pixel")
display(pivot(win_ls, "clear", "median_count"))
display(across_years(win_ls, "clear").round(3))
across_years(win_ls, "clear").to_csv(OUT / "window_across_years_landsat_clear.csv")

## 7. Landsat per-year coverage
Share of AOI pixels with at least one clear observation in the calendar year, and the median per-pixel count (cirrus masked on Landsat 8/9, and the variant that leaves cirrus).

In [ ]:
YEAR = {"calendar_year": tuple(range(1, 13))}
rng = tuple(date.fromisoformat(x) for x in RANGES["landsat"])
dl = inv[inv["sensor"] == "landsat"]
per_year = []
for v in ("clear", "clear_nocirrus"):
    t = inventory.window_summary(
        dl,
        yms["landsat"],
        counts["landsat"][v],
        inside["landsat"],
        YEAR,
        ls_years,
        today=RUN_DATE,
        search_range=rng,
    )
    per_year.append(t.assign(variant=v))
per_year = pd.concat(per_year, ignore_index=True)
plat = ls.groupby(["year", "platform"]).size().unstack(fill_value=0)
tier = ls.groupby(["year", "collection_category"]).size().unstack(fill_value=0).add_prefix("tier_")
wide = per_year[per_year.variant == "clear"].set_index("season_year")[
    ["n_ok", "n_error", "share_ge1_obs", "share_zero_obs", "median_count", "partial_window"]
]
wide = (
    wide.join(
        per_year[per_year.variant == "clear_nocirrus"]
        .set_index("season_year")[["share_ge1_obs", "median_count"]]
        .add_suffix("_nocirrus")
    )
    .join(plat)
    .join(tier)
    .fillna(0)
)
wide.to_csv(OUT / "landsat_per_year.csv")
display(wide.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(wide.index, wide["share_ge1_obs"] * 100)
ax[0].set_ylabel("% of AOI pixels with >= 1 clear obs.")
ax[1].bar(wide.index, wide["median_count"])
ax[1].set_ylabel("median clear obs. per pixel")
for a in ax:
    a.set_xlabel("calendar year")
fig.tight_layout()
fig.savefig(OUT / "landsat_per_year.png", dpi=110)
plt.show()

In [ ]:
timings["total_notebook_s"] = round(
    sum(v for k, v in timings.items() if k.startswith(("search", "audit", "counts"))), 1
)
(OUT / "run_timings.json").write_text(
    json.dumps({"timings_s": timings, "pilot": pilot, "run_date": RUN_DATE.isoformat()}, indent=2)
)
print(json.dumps(timings, indent=2))